# Feature Engineering — raw listings -> scoring schema

Ham veriyi (`data/raw/bursa_listings_synthetic.csv`) `src/scoring.py` + `src/recommender.py`'nin beklediği 0-1 normalize alt skorlara (budget/transport/distance/safety/features/social) çevirir. Gerçek dönüşüm mantığı `src/data_prep.py::engineer_features` ve `build_district_stats` içinde yaşıyor (test edilebilir olması için); bu notebook sadece adım adım gösterir.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))

from src.data_prep import engineer_features, build_district_stats

DATA_DIR = Path("..") / "data"
raw = pd.read_csv(DATA_DIR / "raw" / "bursa_listings_synthetic.csv")
raw.head()

,listing_id,district,room_count,area,building_age,floor,total_floors,price,distance_to_university_km,distance_to_center_km,distance_to_hospital_km,green_space_score,safety_score,public_transport_score,noise_level
0,1,Görükle,3+1,124.8,20,1,7,14860.0,0.0,13.4,6.0,7.4,6.6,6.7,6.1
1,2,Görükle,2+1,75.8,2,2,4,15840.0,0.0,15.0,5.3,6.7,6.7,5.9,6.5
2,3,Görükle,3+1,128.1,2,0,7,18200.0,0.0,14.2,7.3,6.1,6.4,6.2,7.7
3,4,Görükle,3+1,123.2,22,2,6,17170.0,3.3,14.2,5.0,4.9,7.5,7.2,6.3
4,5,Görükle,1+1,48.6,2,8,9,9460.0,0.0,14.8,5.7,6.0,6.4,6.2,7.8


In [2]:
listings = engineer_features(raw)
listings.head()

,district,price,area,room_count,budget,transport,distance,safety,features,social
0,Görükle,14860.0,124.8,3+1,0.813159,0.67,1.000000,0.66,0.616667,0.587300
1,Görükle,15840.0,75.8,2+1,0.794360,0.59,1.000000,0.67,0.714167,0.518544
2,Görükle,18200.0,128.1,3+1,0.749089,0.62,1.000000,0.64,0.826667,0.505422
3,Görükle,17170.0,123.2,3+1,0.768847,0.72,0.875472,0.75,0.555833,0.445422
4,Görükle,9460.0,48.6,1+1,0.916747,0.62,1.000000,0.64,0.901667,0.487764


In [3]:
listings[["budget", "transport", "distance", "safety", "features", "social"]].describe()

,budget,transport,distance,safety,features,social
count,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,0.742262,0.681317,0.663673,0.717383,0.761647,0.645136
std,0.150124,0.163024,0.228580,0.098738,0.135453,0.108182
min,0.000000,0.230000,0.000000,0.460000,0.000000,0.335000
25%,0.663773,0.580000,0.581132,0.650000,0.679167,0.562627
50%,0.768463,0.680000,0.690566,0.720000,0.773333,0.657321
75%,0.853827,0.800000,0.818868,0.790000,0.857500,0.727126
max,1.000000,1.000000,1.000000,1.000000,1.000000,0.901160


In [4]:
district_stats = build_district_stats(listings)
district_stats.sort_values("budget", ascending=False)

,district,avg_price,budget,transport,distance,safety,features,social
1,Görükle,13999.933333,0.829658,0.599867,0.956654,0.657400,0.759694,0.498566
5,Yıldırım,15151.666667,0.807564,0.597867,0.556277,0.600600,0.758439,0.622903
7,Özlüce,16079.133333,0.789773,0.647867,0.777409,0.700200,0.763250,0.635903
3,Mudanya,17307.866667,0.766202,0.402400,0.169132,0.745600,0.758050,0.502280
0,Beşevler,19218.466667,0.729552,0.855400,0.851925,0.746533,0.756294,0.673743
4,Nilüfer,21831.133333,0.679433,0.753200,0.705660,0.799400,0.760428,0.728756
2,Heykel,21870.533333,0.678678,0.899867,0.630566,0.639333,0.778472,0.711472
6,Çekirge,22988.466667,0.657233,0.694067,0.661761,0.850000,0.758544,0.787464


In [5]:
listings.to_csv(DATA_DIR / "processed" / "listings.csv", index=False)
district_stats.to_csv(DATA_DIR / "processed" / "district_stats.csv", index=False)
print("Yazildi:", DATA_DIR / "processed" / "listings.csv", "ve", DATA_DIR / "processed" / "district_stats.csv")

Yazildi: ..\data\processed\listings.csv ve ..\data\processed\district_stats.csv
